# ChaCha20

Jedná se o moderní, rychlý proudový šifrovací algoritmus (stream chipper).

Vnzikla v roce 2008 a vytrvořena byla kryptografem **Danielem J. Bernsteinem** jako vylepšená verze původní *Salsa20*.

## Důvod vzniku

Důvodem byla potřeba po moderní, extrémně rychlé bezpečné šifry, která by byla velmi rychlá výhradně třeba na mobilních zařízeních na softwarové úrovní - totiž konkurenční AES k rychlému běhu vyžaduje speciální hardwarové instrukce procesoru (v té době nebyly přítomny ve všech procesorech). 

Dnes se ChaCha20 řadí mezi nejpoužívanější algoritmy na světě, používá se například pro protokol HTTPS, TLS...

## Pravidla použití

Jak jsem zmínil, jedná se o proudovou šifru - jádrem je generátor pseudonáhodných čísel.

Na začátku si vytváři matici 4x4 o 16 slovech. Do matice se vkládají: pevné konstanty, 256b klíč, počítadlo bloků a nonce.
- Nonce: **N**umber used **once** - nemusí být tajné, jeho úkolem je prostě zajistit jedinečnost zašifrované zprávy

Matice se poté míchá 20 koly (Chacha**20**), která střídají míchaní sloupců a diagonál. Toto míchání funguje na princip ARX (Add, Rotate, XOR). Výsledkém této operace je 64 bajtový nepředvýdatelný proud datm který se dalším XORem spojí s o.t.

## Důsledky porušení kryptografického protokolu

Pro proudové šifry existují dva hlavní implementační problémy:
1. **Znovupoužití Nonce** - Pro stejný klíč nesmí nikdy být použítá stejná nonce - jinak šífra vygeneruje úplně stejný keystream 
    - Pokud by útočník zachytil dvě odlišné začifrované zprávy stejnou noncí, mohl by mezi nimi udělat XOR a získá oba původní o.t.

2. **Chybějící autentizace dat** - ChaCha20 šifruje, ale nechrání integritu
    - Útočník sice nepřečte š.t., ale může v něm změnit po cestě libovolný bit - problém například při převodu peněz..
    - V praxi se vždy používá v kombinaci s autentizačním algoritmem - standard je **ChaCha20-Poly1305**

## Prolomení
ChaCha20 je odolná proti brute force útokům - vyzkoušet všechny možné kombinace 256 bitového klíče je v dnešní době (bez kvantových počítačů) nemožné.
- Fun Fact: Jedná se totiž o možných $ 2^{256} $ různých možností klíče, tedy $ 1,15 \cdot 10^{77} $ možností (V pozorovatelném Vesmíru je zhruba $ 10^{80} $ atomů) 
    - I s miliardou nejvýkonějších počítačů, za předpokladu, že by každý z nich vyzkoušel miliardu kombinací za sekundu, by brute force útok trval **miliardy let**     
    - $ 2^{256} $ je $115\,792\,089\,237\,316\,195\,423\,570\,985\,008\,687\,907\,853\,269\,984\,665\,640\,564\,039\,457\,584\,007\,913\,129\,639\,936$ možných kombinací

Aktuální pokusy o prolomení ChaCha20 jsou založeny na matematických útocích na redukované verzi ChaCha20 (která dělá méně míchání). Pro 20 kolovou verzi neexistuje matematícký způsob, jak šifru prolomit.

Zajímavostí je, že algoritmus je i **imunní vůči** tzv. *"side-channel útokům"* - tedy nejde z procesoru vypozorovat, jak šifra funguje, jak dlouho trvá, jak je náročná...
- Útočník pro pokus rozluštit šifru často analyzují:
    - Časvání - Měření na milisekundy přesně, jak dlouho trvá zašifrovat zprávu (Šifrování z "A" trvá o 1ms déle, než z "B" a útočník dokáže rozluštit šifru)
    - Spotřeba energie - Útočník měří, kolik elektřiny procesor odebírá - zpracování 1 je náročnější než 0
    - Hluk - Útočník mikrofonem měří pískání cívky v PC, která mění frekvenci podle aktuálních výpočtů 

---
---

## Kód

## 1. State inicialization

Tato část kódu se zabývá přípravou klíčových součástí šifry do podoby, která bude využita k šifrování.

Vytvoří se vlastně taková matice 4x4, první 4 hodnoty jsou konstantní, ty se získávají z textu *"expand 32-byte k"* (převedených do ASCII), poté následuje klíč, ten se ale před vložením musí rozdělit na 8 částí po 4 bytech, poté následuje count, a nakonec Nonce, který se také musí rozdělit, ale na 3 skupiny po 4 bytech.

Výsledek je seznam, který má dohromady 16 prvků, tedy 64 bajtů.   

In [ ]:
def create_matrix(key :bytes, nonce :bytes, count :int) -> List:

    c0 = 0x61707865
    c1 = 0x3320646e
    c2 = 0x79622d32
    c3 = 0x6b206574

    m_list = [c0, c1, c2, c3]

    # TODO: Optimalizuj 
    for i in range(0, len(key), 4):
        skupiny = key[i : i+4]
        prevod_cislo = int.from_bytes(skupiny, "little")
        m_list.append(prevod_cislo)

    m_list.append(count)

    for i in range(0, len(nonce), 4):
        skupiny = nonce[i : i+4]
        prevod_cislo = int.from_bytes(skupiny, "little")
        m_list.append(prevod_cislo)
    
    return m_list


## 2. Quarter round

### 2.1 Funkce rotl32(x, n)
Tato funkce se stará o posun (rotaci) bitů, tedy pokud bit "vypadne" zprava, musí objevit znovu zleva.

### 2.2 Funkce uprava_indexu
Tato funkce dělá klíčovou součást ChaCha20 - vezme 4 indexy z matice, kterou jsme si připravili a začne je upravovat na základě principu ARX

Funkce:
1. Na index "A" se uloží sečtení hodnoty z indexu "A" a "B" a poté se udělá "kontrola hranice", tedy operace AND s 32 bity jen 1 - pokud by výsledek operace sečtení indexů byl víc než 32 bitů, zbytek se zahodí
2. Zde se dělá XOR indexu "C" a indexu "A"
3. Udělá se rotace - 16 bitů zleva se posune do prava 

In [ ]:
def rotl32(x, n):
    return ((x << n) & 0xffffffff) | (x >> (32 - n))

def uprava_indexu(matice :list, index0 :int, index1 :int, index2 :int, index3 :int):

    # ARX 

    # a = (a + b) % 2^32, d je xor a 
    matice[index0] = (matice[index0] + matice[index1]) & 0xffffffff # uprava prvniho indexu
    matice[index3] = matice[index3] ^ matice[index0] # xor
    matice[index3] = rotl32(matice[index3], 16) # rotace bitů   
    
    # c = (c + b) % 2^32
    matice[index2] = (matice[index2] + matice[index3]) & 0xffffffff
    matice[index1] = matice[index1] ^ matice[index2]
    matice[index1] = rotl32(matice[index1], 12)

    matice[index0] = (matice[index0] + matice[index1]) & 0xffffffff 
    matice[index3] = matice[index3] ^ matice[index0] 
    matice[index3] = rotl32(matice[index3], 8)    
    
    matice[index2] = (matice[index2] + matice[index3]) & 0xffffffff
    matice[index1] = matice[index1] ^ matice[index2]
    matice[index1] = rotl32(matice[index1], 7)

##  3. ChaCha20 block

Zde se odehrává samotný algoritmus dané šifry. 

Nejdříve je nutné vytvořit si kopii matice, aby po úpravě bylo možné pracovat i s původními daty.

Poté se začnou dělat samotné rundy - 20 rund (vlastně to jsou double rundy), volání funkce uprava_indexu na sloupce a poté na diagonály.

To co se děje průběžná transformace všech hodnot v dané matici.

Nakonec se nové výsledky (hodnoty) sečtou se starými a znovu se provede kontrola přetečení. 

In [ ]:
def chacha(klic :bytes, nonce :bytes, count :int):

    matr = create_matrix(klic, nonce, count)
    matr_cp = list(matr)

    for i in range(0, 10):
        for x in range(4):
            # sloupce
            uprava_indexu(matr, x, x+4, x+8, x+12)

            # diag
            uprava_indexu(matr, x, 4+(x+1)%4, 8+(x+2)%4, 12+(x+3)%4)

    # secteni s puvodnimi hodnotami s modulem!

    vysledek = bytearray()
    for i in range(0, len(matr)):
        matr[i] = matr[i] + matr_cp[i] & 0xffffffff

    for cislo in matr:
        vysledek.extend(cislo.to_bytes(4, "little"))

    return vysledek

## 4. (De)šifrování

V této fázi už se zpráva šifruje - vezme se bajt zprávy a ten se zašifruje s bajtem zašifrovaného klíče

Tenhle kód funguje jak pro šifrování, tak i pro dešifrování, protože XOR funguje oboustranně.

In [ ]:
def sifrovani(klic, zprava):
    
    zasifrovany_text = bytearray()
    for i in range(0, len(zprava)):
        zasifrovany_text.append(zprava[i] ^ klic[i])

    return bytes(zasifrovany_text)

## Test

In [ ]:
count = 1
uz_input = ""
uz_input = input("Zadejte text k šifrování: ")
zprava = uz_input.encode('utf-8')

if (len(zprava)) >= 64:
    if len(zprava) % 64  == 0:
        rozdil = len(zprava) / 64
        count += rozdil
    else:
        count += (len(zprava) // 64) + 1 


klic = b'\x00\x01\x02\x03\x04\x05\x06\x07\x08\x09\x0a\x0b\x0c\x0d\x0e\x0f\x10\x11\x12\x13\x14\x15\x16\x17\x18\x19\x1a\x1b\x1c\x1d\x1e\x1f'
nonce = b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x02'

cely_keystream = bytearray()
for i in range(0, count):
    kus_ks = chacha(klic, nonce, i)  
    cely_keystream.extend(kus_ks)

sifra = sifrovani(cely_keystream, zprava)
print(f"Šifrovaný text (hex): {sifra.hex()}")

desifrovany_text = sifrovani(cely_keystream, sifra)
print(f"Dešifrovaný text: {desifrovany_text.decode('utf-8')}")

## Upravený klíč

In [ ]:
klic = b'\xff\x01\x02\x03\x04\x05\x06\x07\x08\x09\x0a\x0b\x0c\x0d\x0e\x0f\x10\x11\x12\x13\x14\x15\x16\x17\x18\x19\x1a\x1b\x1c\x1d\x1e\x1f'
spatny_keystream = bytearray()
for i in range(0, count):
    kus_ks = chacha(klic, nonce, i)  
    spatny_keystream.extend(kus_ks)

spatna_sifra = sifrovani(cely_keystream, zprava)
print(spatna_sifra.decode('utf-8', errors='replace'))